# 17 — Structured-data RAG: SQL calculations plus policy evidence

**Goal:** answer a business question using actual SQLite rows, exact arithmetic, and policy documents.
**Prerequisites:** notebooks 01–05; basic Python and SQL concepts. **Time:** 45–60 minutes.

We build `question → validated query plan → parameterized SQL → rows + policy retrieval → Claude`.
The database is fictional, local, and created under `artifacts/structured/`. Queries perform real
joins and aggregations. Claude chooses a reviewed query template rather than receiving write access.
A separate inspection cell exposes the parser and database enforcement used by those templates.

**Mode:** live mode uses the existing `CLAUDE_KEY` for a plan and an answer (normally two cached calls).
`RAG_OFFLINE=1` uses an explicitly labeled prepared plan, executes the real SQL, and skips generation.
Optional free-form SQL generation is deliberately outside this lesson's narrower query-plan interface.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the repository or in notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "rag_workshop").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the LangchainPractical project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rag_workshop import (
    Document, load_documents, chunk_documents, TfidfIndex, DenseIndex,
    bm25_search, reciprocal_rank_fusion, context_text, ask_claude,
    rag_answer, show_hits, settings, offline_mode,
)
import pandas as pd
from IPython.display import display, Markdown

print("Mode:", "OFFLINE: Claude calls skipped" if offline_mode() else "LIVE: Claude calls use API credits")
print("Claude model:", settings().model)


## 1. Create a small relational dataset

Tables represent plans, customer accounts, individual seats, and an order ledger. They are distinct:
a purchased seat can be inactive, and a pending order is not paid revenue. Enterprise pricing is `NULL`
because the policy requires a custom quote. Team is **INR 2,400 per user per month, billed annually,
with a minimum of 5 users**. All calculations exclude additional taxes.

Creating the fixture is the only write step. The query helper opens a new read-only connection for
each request. Rerunning this cell reuses the fixture; the creator refuses to overwrite an existing file.


In [ ]:
import json
import sqlite3
from rag_workshop.structured import (
    TABLE_COLUMNS, QUERIES, PLAN_HELP, create_demo_database, validate_plan, parse_plan_response,
    validate_select, safe_select, execute_plan, sql_evidence,
)

database_path = ROOT / "artifacts/structured/northstar_demo.sqlite"
if not database_path.exists():
    create_demo_database(database_path)
print("Database:", database_path.relative_to(ROOT))
display(pd.DataFrame([{"table": table, "columns": ", ".join(sorted(columns))}
                      for table, columns in TABLE_COLUMNS.items()]))
display(pd.DataFrame(safe_select(database_path, "SELECT * FROM accounts ORDER BY account_id")["rows"]))
display(pd.DataFrame(safe_select(database_path, "SELECT * FROM plans ORDER BY plan_id")["rows"]))


## 2. Start with a visible join and aggregate

The query joins accounts to plans and active seats. `COUNT` counts active assignments; subtraction
computes unused purchased users. SQLite calculates the annual list price from the plan rate, seat
minimum, and 12-month term. It does not estimate an Enterprise contract value.

Read the SQL before running it. `:account_name` is a bound value, not a string pasted into SQL.
Expected for Cedar Labs: 8 purchased, 6 active, 2 unused, and INR 230,400 annual list price before tax.


In [ ]:
print(QUERIES["account_overview"])
cedar_plan = {"query_id": "account_overview", "parameters": {"account_name": "Cedar Labs"}}
cedar = execute_plan(database_path, cedar_plan)
display(pd.DataFrame(cedar["rows"]))
row = cedar["rows"][0]
assert (row["purchased_users"], row["active_users"], row["unused_users"]) == (8, 6, 2)
assert row["annual_list_price_inr_ex_tax"] == 230400


## 3. Keep totals, units, and unknown values explicit

Group accounts by region and plan. Splitting the plan dimension prevents an unknown Enterprise value
from being silently treated as zero in a combined sum. The order ledger is a different measure: the
query includes only rows marked `paid`, irrespective of the forward-looking annual list quote.
`NULL` is unknown; an empty result is absence of matching rows, not proof of a zero balance.


In [ ]:
region_result = execute_plan(database_path, {"query_id": "region_summary", "parameters": {}})
display(pd.DataFrame(region_result["rows"]))
paid = execute_plan(database_path, {"query_id": "paid_orders", "parameters": {"account_name": "Cedar Labs"}})
display(pd.DataFrame(paid["rows"]))
assert paid["rows"][0]["paid_order_count"] == 2
assert paid["rows"][0]["paid_order_total_inr_ex_tax"] == 460800
enterprise = execute_plan(database_path, {"query_id": "account_overview",
                                          "parameters": {"account_name": "Orion Systems"}})
assert enterprise["rows"][0]["annual_list_price_inr_ex_tax"] is None
print("Enterprise list price:", enterprise["rows"][0]["annual_list_price_inr_ex_tax"], "(unknown)")


## 4. Natural language → a constrained query plan

Instead of accepting arbitrary generated SQL, ask Claude to choose a query ID and typed parameters.
The model has four supported intents and an `unsupported` outcome. This is a useful form of structured
retrieval: flexible wording at the input, a narrow auditable interface at the execution boundary.

In offline mode the next object is a **prepared gold plan**, not model output. In live mode it is parsed
from Claude's response. The parser accepts JSON or one fenced JSON object; explanatory prose outside
that one fence is ignored, never executed. Multiple plans or a malformed schema stop before execution;
they are never silently replaced by a guess. Inspect the accepted plan before running the next cell.


In [ ]:
question = "For Cedar Labs, how many purchased users are unused, what is the annual Team list cost, and is tax included?"
if offline_mode():
    proposed_plan = {"query_id": "account_overview", "parameters": {"account_name": "Cedar Labs"}}
    print("OFFLINE: using a prepared gold query plan, not Claude output.")
else:
    plan_response = ask_claude("Question: " + question, system=PLAN_HELP, max_tokens=300)
    proposed_plan = parse_plan_response(plan_response["text"])
query_id, parameters = validate_plan(proposed_plan)
print(json.dumps(proposed_plan, indent=2))
print("Selected SQL:", QUERIES.get(query_id, "No SQL: unsupported intent"))


## 5. Inspect the read-only execution boundary

The reusable helper is small enough to read. It parses exactly one allowed `SELECT` with SQLGlot,
rejects unknown tables and unsupported syntax, then opens SQLite with `mode=ro` and `query_only`.
An authorizer independently permits only approved table columns and a short function allowlist.
Bound parameters, output-row limits, and a progress-handler deadline/VM-step budget add separate controls.

This intentionally excludes CTEs, subqueries, arbitrary functions, and writable queries. These controls
do not make a model's chosen intent correct; compare the plan with the user's actual question.
A production system also needs real user/tenant permissions, audited views, and sensitive-column policies.


In [ ]:
import inspect
print(inspect.getsource(validate_select))
print(inspect.getsource(safe_select))
result = execute_plan(database_path, proposed_plan, max_rows=20, timeout_s=1, max_vm_steps=100000)
if result is None:
    print("Unsupported intent: no database query executed.")
else:
    display(pd.DataFrame(result["rows"]))
    print("Citation:", result["id"], "| Truncated:", result["truncated"])


## 6. Retrieve the policy explanation separately

SQL supplies exact record values and arithmetic. The policy document explains annual billing, the
minimum seat count, taxes, and custom Enterprise pricing. Search only the relevant public plan corpus;
do not expose unrelated internal documents while answering a customer pricing question.
The snapshot's `policy_source` also provides a deterministic link back to `product_plans`.


In [ ]:
policy_docs = [d for d in load_documents() if d.id in {"product_plans", "refunds", "data_retention"}
               and d.metadata.get("visibility") == "public"]
policy_index = TfidfIndex(policy_docs)
policy_hits = policy_index.search("Team plan annual billing minimum users taxes Enterprise custom pricing", k=1)
display(show_hits(policy_hits))
assert policy_hits[0]["id"] == "product_plans"
row_hits = sql_evidence(result)
# Do not answer an account-specific question from policy alone when no rows exist.
combined_hits = row_hits + policy_hits if row_hits else []
print(context_text(combined_hits, max_chars=6000))


## 7. Combine structured and unstructured evidence with Claude

Ask the generator to use the computed values rather than perform fresh arithmetic. SQL citations
identify the query, bound parameters, and result snapshot; policy citations identify the source text.
This distinction lets a viewer audit both the calculation and its business meaning.

**Expected facts:** 2 unused purchased users; INR 230,400 for one annual term at the advertised Team
list rate; taxes additional. This is a list-price calculation, not a payment demand or customer contract.


In [ ]:
answer = rag_answer(question + " Use the SQL's computed numbers without recalculating. "
                    "Distinguish annual list price from actual paid orders or negotiated contract price. "
                    "Cite SQL evidence for record values and policy evidence for billing/tax conditions.",
                    combined_hits, max_tokens=500)
display(Markdown(answer["text"]))
print({k: answer[k] for k in ["offline", "input_tokens", "output_tokens", "cached"]})


## 8. Try unsafe and unsupported inputs

Do not trust a `SELECT` prefix or search for forbidden words with a regex. A parsed statement boundary,
a constrained grammar, and database enforcement are separate checks. The fixture makes it safe to
demonstrate rejected writes, attachment attempts, file functions, and multi-statement input.
The quoted attack string in a bound account name is treated literally and matches no record.


In [ ]:
rejected_queries = [
    "DELETE FROM accounts",
    "SELECT * FROM accounts; DROP TABLE accounts",
    "PRAGMA writable_schema=ON",
    "ATTACH DATABASE 'other.sqlite' AS other",
    "SELECT load_extension('example')",
    "SELECT readfile('private.txt')",
    "SELECT * FROM sqlite_master",
]
rejection_rows = []
for sql in rejected_queries:
    try:
        safe_select(database_path, sql)
        raise AssertionError("Unexpectedly accepted unsafe SQL")
    except (ValueError, sqlite3.DatabaseError) as exc:
        rejection_rows.append({"input": sql, "outcome": type(exc).__name__})
display(pd.DataFrame(rejection_rows))
injection_plan = {"query_id": "account_overview",
                  "parameters": {"account_name": "Cedar Labs' OR 1=1 --"}}
assert execute_plan(database_path, injection_plan)["rows"] == []
assert execute_plan(database_path, {"query_id": "unsupported", "parameters": {}}) is None
assert safe_select(database_path, "SELECT COUNT(*) AS n FROM accounts")["rows"][0]["n"] == 4
print("Bound text did not become SQL; all four accounts still exist.")


## 9. Handle row limits, empty results, and resource budgets

A truncated result must not be described as all records. We fetch at most one extra row to detect
truncation; SQL aggregation happens before that output cap. A row cap alone does not limit query
computation, so an instruction budget also interrupts expensive joins. Unknown account names abstain.


In [ ]:
limited = safe_select(database_path, "SELECT * FROM accounts ORDER BY account_id", max_rows=2)
assert limited["truncated"] is True and len(limited["rows"]) == 2
print("Two-row preview; truncated:", limited["truncated"])
missing = execute_plan(database_path, {"query_id": "account_overview",
                                       "parameters": {"account_name": "Unknown Customer"}})
assert sql_evidence(missing) == []
print(rag_answer("What does Unknown Customer owe?", sql_evidence(missing))["text"])
try:
    safe_select(database_path,
                "SELECT COUNT(*) AS n FROM seats AS a CROSS JOIN seats AS b CROSS JOIN seats AS c",
                max_vm_steps=100)
    raise AssertionError("Budget should have interrupted the expensive query")
except sqlite3.OperationalError as exc:
    print("Compute budget enforced:", str(exc))


## 10. Practice: minimum users, group totals, and unsupported questions

1. Request a Team quote for 3 users. Apply the 5-user minimum and calculate the annual total in SQL.
2. Find the West-region Team purchased-user total and annual list-price total.
3. Explain why Orion's unknown Enterprise price must stay `NULL` instead of becoming zero.
4. Propose a plan for “predict next year's profit.” It must be `unsupported`: no forecast data is present.

**Expected:** 5 billable users → INR 144,000/year before tax; West Team → 20 purchased users and
INR 576,000 annual list price before tax. The solution performs no additional Claude calls.


In [ ]:
quote = execute_plan(database_path, {"query_id": "team_quote", "parameters": {"requested_users": 3}})
display(pd.DataFrame(quote["rows"]))
assert quote["rows"][0]["billable_users"] == 5
assert quote["rows"][0]["annual_list_price_inr_ex_tax"] == 144000
west = next(r for r in region_result["rows"] if r["region"] == "West" and r["plan"] == "Team")
assert (west["purchased_users"], west["annual_list_price_inr_ex_tax"]) == (20, 576000)
unsupported_plan = {"query_id": "unsupported", "parameters": {}}
assert execute_plan(database_path, unsupported_plan) is None
print("West-region result:", west)
print("Unsupported questions execute no SQL and produce no invented forecast.")


## Primary references and production extensions

- [Python sqlite3: parameters, URI connections, authorizer and progress handler](https://docs.python.org/3/library/sqlite3.html)
- [SQLite read-only URI mode](https://www.sqlite.org/uri.html)
- [SQLite authorizer API](https://www.sqlite.org/c3ref/set_authorizer.html)
- [SQLGlot parser documentation](https://sqlglot.com/sqlglot.html)

Add approved query templates with tests for cardinality, NULLs, duplicate joins, units, date filters,
and time zones. For a broader SQL agent, validate schema references and user permissions, limit cost,
review query meaning, and evaluate on held-out questions. A syntactically safe query can still answer
the wrong question. Refresh structured and document snapshots together so policy versions stay aligned.
